In [99]:
import ast
import numpy as np
import pandas as pd

In [100]:
jobs_df = pd.read_excel("final_jobs_fixed.xlsx", engine="openpyxl")
jobs_df.head()

,id,title,company,location,country,salary_min,salary_max,contract_type,redirect_url,description,created
0,5753281217,Data Analyst (m/w/d),"{'display_name': 'STRABAG BRVZ GMBH', '__CLASS...","{'area': ['Österreich'], '__CLASS__': 'Adzuna:...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5753281217?se=_l...,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05T16:40:19Z
1,5753281216,Data Analyst (m/w/d),{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name': 'Spittal an der Drau', '__CLA...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5753281216?se=_l...,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05T16:40:19Z
2,5762094132,Data Analyst - MS PowerBI (m/w/d),{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['Österreich', 'Wien'], '__CLASS__': ...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5762094132?se=_l...,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13T06:58:14Z
3,5727781690,Data Analyst (m/w/d),{'display_name': 'ÖSW Österreichisches Siedlun...,"{'display_name': 'Josefstadt, Wien', 'area': [...",at,NaN,NaN,permanent,https://www.adzuna.at/details/5727781690?utm_m...,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13T08:03:25Z
4,5758608143,Data Analyst,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name': 'Österreich', 'area': ['Öster...",at,NaN,NaN,NaN,https://www.adzuna.at/details/5758608143?utm_m...,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10T17:18:19Z


In [101]:
jobs_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6718 entries, 0 to 6717
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             6718 non-null   int64  
 1   title          6718 non-null   object 
 2   company        6718 non-null   object 
 3   location       6718 non-null   object 
 4   country        6718 non-null   object 
 5   salary_min     2986 non-null   float64
 6   salary_max     2982 non-null   float64
 7   contract_type  1700 non-null   object 
 8   redirect_url   6718 non-null   object 
 9   description    6718 non-null   object 
 10  created        6718 non-null   object 
dtypes: float64(2), int64(1), object(8)
memory usage: 577.5+ KB


In [102]:
import re

def clean_and_extract_research_data(row):
    # Initialize all required target fields for answering the research questions
    country = "Unknown"
    salary = "Not Found"
    work_mode = "Unspecified"
    seniority = "Junior/Mid"
    skills = []

    # Fetch and safely convert to lowercase strings to prevent exceptions from missing values (NaN)
    desc = str(row["description"]).lower() if pd.notna(row["description"]) else ""
    title = str(row["title"]).lower() if pd.notna(row["title"]) else ""

    # Extract and format country code/name if available
    if pd.notna(row["country"]):
        country = str(row["country"]).upper().strip()

    # Process salary information
    # Check if structured minimum and maximum salary fields both exist

    if pd.notna(row["salary_min"]) and pd.notna(row["salary_max"]):
        # Combine them into a standardized range string (e.g., "40000-60000")
        salary = f"{int(row['salary_min'])}-{int(row['salary_max'])}"
    else:
        # If structured data is missing, fallback to extracting salary from the description using Regex
        # Regex pattern to match currency symbols (£, $, €) or 'k' suffixes, along with range indicators (-, ~, to)
        salary_pattern = r"([£$€]\s*\d+[\d,\s]*[-\s~toTO]?[\d,\s]*|\d+\s*[kK][-\s~toTO]?\d+\s*[kK])"
        salary_match = re.search(salary_pattern, desc)
        if salary_match:
            # If a match is found, extract and trim the matched string

            salary = salary_match.group(0).strip()

    # Define a list of core technical skills to scan for in the description 
    skill_keywords = [
        "python",
        "sql",
        "excel",
        "tableau",
        "power bi",
        "r",
        "sas",
        "aws",
        "azure",
        "spark",
    ]
    # Skill Extraction
    # Loop through the predefined skill keywords and check if they appear in the description or title

    for skill in skill_keywords:
        if skill in desc or skill in title:
            # Convert to uppercase (e.g., 'python' -> 'PYTHON') and append to the skills list
            skills.append(skill.upper())
    # Join the extracted skills with commas, or assign "None Mentioned" if the list is empty        
    extracted_skills = ", ".join(skills) if skills else "None Mentioned"

    # Work Mode Classification
    # Classify the work mode based on standard keywords found in the description 

    if "remote" in desc or "home-based" in desc or "home based" in desc:
        work_mode = "Remote"
    elif "hybrid" in desc or "flexible" in desc:
        work_mode = "Hybrid"
    elif "onsite" in desc or "office" in desc:
        work_mode = "Onsite"

    # Seniority Level Classification
    # Classify career level by scanning for tier-specific keywords in the description or title
     
    if "senior" in desc or "sr" in title or "lead" in desc or "principal" in desc:
        seniority = "Senior"
    elif "junior" in desc or "jr" in title or "entry" in desc or "trainee" in desc:
        seniority = "Junior"
    elif "intern" in desc or "praktikum" in desc:
        seniority = "Intern"
    elif "middle" in desc or "mid" in title or "entry" in desc:
        seniority = "Middle"
    else:
        # Fallback category if no specific seniority keywords are detected
        seniority = "Unspecified"

    # Return Processed Data
    # Pack all refined variables into a Pandas Series to easily append back to the DataFrame
    return pd.Series([country, salary, extracted_skills, work_mode, seniority])


jobs_df["created"] = pd.to_datetime(jobs_df["created"], errors="coerce", utc=True).dt.tz_localize(None)

# Apply the cleaning function row-by-row (axis=1) and assign the outputs to 5 new columns 
jobs_df[
    [
        "clean_country",
        "clean_salary",
        "extracted_skills",
        "work_mode",
        "seniority_level",
    ]
] = jobs_df.apply(clean_and_extract_research_data, axis=1)

# Convert the creation timestamp into a standardized 'Year-Week' period string (e.g., '2026W24') for weekly analysis 
jobs_df["year_week"] = jobs_df["created"].dt.to_period("W").astype(str)


In [103]:
jobs_df['clean_country'].value_counts()

clean_country
FR    2095
GB    1968
DE     782
PL     591
NL     442
IT     271
ES     245
BE     191
CH      68
AT      65
Name: count, dtype: int64

In [104]:
# Create a pivot table to count the number of jobs ('id') by country and work mode
pivot_country = jobs_df.pivot_table(
    index="clean_country",
    columns="work_mode",
    values="id",
    aggfunc="count",
)

# Fill any missing/NaN values with 0 (indicating zero jobs found for that combination) 
pivot_country = pivot_country.fillna(0)


pivot_country

work_mode,Hybrid,Onsite,Remote,Unspecified
clean_country,,,,
AT,24,9,7,25
BE,51,17,20,103
CH,17,12,12,27
DE,319,69,242,152
ES,98,30,37,80
FR,283,186,108,1518
GB,651,192,292,833
IT,23,74,26,148
NL,93,26,57,266


In [105]:
# Create a pivot table to count the number of jobs ('id') by country and seniority level
pivot_seniority_leve = jobs_df.pivot_table(
    index="clean_country",
    columns="seniority_level",
    values="id",
    aggfunc="count",
)

# Fill any missing/NaN values with 0 to ensure clean numeric analysis
pivot_seniority_leve = pivot_seniority_leve.fillna(0)

pivot_seniority_leve

seniority_level,Intern,Junior,Middle,Senior,Unspecified
clean_country,,,,,
AT,22.0,4.0,1.0,20.0,18.0
BE,56.0,2.0,0.0,80.0,53.0
CH,18.0,3.0,0.0,30.0,17.0
DE,270.0,21.0,0.0,294.0,197.0
ES,25.0,5.0,1.0,169.0,45.0
FR,404.0,48.0,1.0,1186.0,456.0
GB,113.0,33.0,1.0,1145.0,676.0
IT,48.0,15.0,0.0,184.0,24.0
NL,85.0,34.0,0.0,190.0,133.0


In [106]:
# Group the dataset by the 'year_week' column and calculate the total number of records (rows) per week
jobs_df.groupby('year_week').size()

year_week
2022-06-27/2022-07-03       4
2022-08-29/2022-09-04       1
2022-10-24/2022-10-30       1
2022-11-28/2022-12-04       2
2023-02-13/2023-02-19       3
                         ... 
2026-05-18/2026-05-24     633
2026-05-25/2026-05-31     623
2026-06-01/2026-06-07    1029
2026-06-08/2026-06-14    1279
2026-06-15/2026-06-21     337
Length: 113, dtype: int64

In [107]:
# Expand the comma-separated skills into individual rows
# .assign(): Split the comma-separated string 'extracted_skills' into Python lists (e.g., "PYTHON, SQL" -> ["PYTHON", "SQL"])
# .explode(): Flatten the lists so each skill gets its own dedicated row, duplicating other row values accordingly
exploded_df = jobs_df.assign(extracted_skills=jobs_df["extracted_skills"].str.split(", ")).explode("extracted_skills")
# Use pivot_table to achieve the same cross-tabulation frequency count
pivot_skills = exploded_df.pivot_table(
    index="country", 
    columns="extracted_skills", 
    values="id", 
    aggfunc="count"
)

# Fill NaN with 0 since missing combination means 0 occurrences
pivot_skills = pivot_skills.fillna(0)
# Convert all counts from float to integer for a cleaner, non-decimal display
pivot_skills = pivot_skills.astype(int)

pivot_skills

extracted_skills,AWS,AZURE,EXCEL,None Mentioned,POWER BI,PYTHON,R,SAS,SPARK,SQL,TABLEAU
country,,,,,,,,,,,
at,9,15,24,0,22,23,65,0,7,37,11
be,23,31,81,0,68,71,191,11,13,98,39
ch,12,13,29,0,22,34,68,2,9,46,15
de,96,208,208,0,319,286,782,15,55,482,121
es,29,23,156,0,78,100,245,55,12,155,63
fr,216,198,907,0,670,689,2095,108,135,1006,743
gb,207,202,889,1,472,497,1967,61,110,721,224
it,24,30,159,0,106,90,271,18,18,132,50
nl,32,61,126,0,131,152,442,9,36,185,41


In [108]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6718 entries, 0 to 6717
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                6718 non-null   int64         
 1   title             6718 non-null   object        
 2   company           6718 non-null   object        
 3   location          6718 non-null   object        
 4   country           6718 non-null   object        
 5   salary_min        2986 non-null   float64       
 6   salary_max        2982 non-null   float64       
 7   contract_type     1700 non-null   object        
 8   redirect_url      6718 non-null   object        
 9   description       6718 non-null   object        
 10  created           6718 non-null   datetime64[ns]
 11  clean_country     6718 non-null   object        
 12  clean_salary      6718 non-null   object        
 13  extracted_skills  6718 non-null   object        
 14  work_mode         6718 n

In [109]:
# Define a function to parse city names from a location/display name string
def extract_city_simple(location_str):
    # Check for invalid types (non-strings) or empty/whitespace-only entries
    if not isinstance(location_str, str) or not location_str.strip():
        return "Unknown"

    # Use regular expression to extract the value associated with the 'display_name' key    
    match = re.search(r"'display_name':\s*'([^']+)'", location_str)
    if match:

        # Return the captured group (the actual city/location name)
        return match.group(1)
    return "Unknown"


# Standardize country codes by converting to uppercase, stripping whitespace, and filling NaNs
jobs_df["clean_country"] = (
    jobs_df["country"].str.upper().str.strip().fillna("UNKNOWN")
)
# Apply the custom extraction function row-by-row to extract clean city names
jobs_df["clean_city"] = jobs_df["location"].apply(extract_city_simple)



# Create a baseline aggregation table counting job volume by country and city 
city_table = (
    jobs_df.groupby(["clean_country", "clean_city"])
    .size()
    .reset_index(name="job_count")
)

# Calculate the percentage market share of each city within its respective country
# .transform() scales the group total back to the original dataframe size to allow direct division
city_table["percentage_in_country"] = city_table.groupby("clean_country")[
    "job_count"
].transform(lambda x: (x / x.sum()) * 100)

# Extract the top 3 cities with the highest job counts for each country 
final_table = (
    # Sort primarily by country (A-Z) and secondarily by job count descending (highest first)
    city_table.sort_values(
        ["clean_country", "job_count"], ascending=[True, False]
    )
    # Group by country again to isolate each regional market
    .groupby("clean_country")
    # Slice the top 3 rows for each country group
    .head(3)
)

# Round the calculated percentage values to 2 decimal places for cleaner reporting 
final_table["percentage_in_country"] = final_table[
    "percentage_in_country"
].round(2)


# Extract company names from the nested company/display_name field
def extract_company_display_name(company_value):
    if isinstance(company_value, dict):
        return company_value.get("display_name", "Unknown")

    if pd.isna(company_value):
        return "Unknown"

    company_text = str(company_value)
    try:
        company_dict = ast.literal_eval(company_text)
        if isinstance(company_dict, dict):
            return company_dict.get("display_name", "Unknown")
    except (ValueError, SyntaxError):
        pass

    match = re.search(r"'display_name':\s*'([^']+)'", company_text)
    return match.group(1) if match else "Unknown"


jobs_df["clean_company"] = jobs_df["company"].apply(extract_company_display_name)

 
print(final_table)


     clean_country                          clean_city  job_count  \
15              AT                          Österreich         20   
13              AT                    Wien, Österreich         16   
4               AT                  Innere Stadt, Wien          9   
21              BE                              België        127   
23              BE          Brussel, Brussel Hoofdstad         13   
36              BE                            Mechelen          6   
61              CH                             Schweiz         22   
66              CH                              Zürich         15   
46              CH               Bern, Bern-Mittelland          5   
142             DE                         Deutschland        122   
113             DE                 Berlin, Deutschland         82   
202             DE                Hamburg, Deutschland         22   
389             ES                              España         89   
394             ES                

In [110]:
def extract_company_display_name(company_value):
    if pd.isna(company_value):
        return "Unknown"

    company_text = str(company_value)

    match = re.search(r"'display_name':\s*'([^']+)'", company_text)

    if match:
        return match.group(1)

    return "Unknown"


jobs_df["clean_company"] = jobs_df["company"].apply(extract_company_display_name)

jobs_df[["company", "clean_company"]].head()

,company,clean_company
0,"{'display_name': 'STRABAG BRVZ GMBH', '__CLASS...",STRABAG BRVZ GMBH
1,{'__CLASS__': 'Adzuna::API::Response::Company'...,STRABAG BRVZ GMBH
2,{'__CLASS__': 'Adzuna::API::Response::Company'...,Baustoff + Metall Gesellschaft m.b.H. Österrei...
3,{'display_name': 'ÖSW Österreichisches Siedlun...,ÖSW Österreichisches Siedlungswerk Gemeinnützi...
4,{'__CLASS__': 'Adzuna::API::Response::Company'...,Hitapps


In [111]:
jobs_df.sample(20)

,id,title,company,location,country,salary_min,salary_max,contract_type,redirect_url,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,year_week,clean_city,clean_company
972,5735926994,Ingénieur Data Analyst Supply S&OP F/H,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name': 'Cher, Centre', '__CLASS__': ...",fr,NaN,NaN,permanent,https://www.adzuna.fr/details/5735926994?utm_m...,Le poste de Ingénieur Data Analyst Supply S&OP...,2026-05-21 05:44:26,FR,Not Found,"EXCEL, TABLEAU, POWER BI, R",Unspecified,Senior,2026-05-18/2026-05-24,"Cher, Centre",MBDA
2331,5761802611,Analyste pricing H/F,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['France'], '__CLASS__': 'Adzuna::API...",fr,NaN,NaN,NaN,https://www.adzuna.fr/details/5761802611?utm_m...,Informations générales\nEntité de rattachement...,2026-06-12 23:08:58,FR,Not Found,"EXCEL, TABLEAU, R",Unspecified,Senior,2026-06-08/2026-06-14,France,Legallais
1069,5745816338,Data analyst et chargé d'études sur le médicam...,"{'display_name': 'Ministères de la Santé, des ...","{'display_name': '15ème Arrondissement, Paris'...",fr,NaN,NaN,contract,https://www.adzuna.fr/details/5745816338?utm_m...,Vos missions en quelques mots\nEn tant que dat...,2026-05-30 22:44:50,FR,Not Found,R,Unspecified,Senior,2026-05-25/2026-05-31,"15ème Arrondissement, Paris","Ministères de la Santé, des Solidarités et du ..."
691,5321338818,Consultant Data Analyst - Niji H/F,{'__CLASS__': 'Adzuna::API::Response::Company'...,{'__CLASS__': 'Adzuna::API::Response::Location...,fr,NaN,NaN,NaN,https://www.adzuna.fr/details/5321338818?utm_m...,Le pôle Data de Niji c'est avant tout une équi...,2025-07-25 09:22:02,FR,Not Found,"PYTHON, SQL, TABLEAU, POWER BI, R, SPARK",Unspecified,Senior,2025-07-21/2025-07-27,"Issy-les-Moulineaux, Boulogne-Billancourt",Niji
5474,5754609724,Data Governance Analyst,"{'display_name': 'British Gas', '__CLASS__': '...","{'display_name': 'Hamilton, South Lanarkshire'...",gb,42862.04,42862.04,permanent,https://www.adzuna.co.uk/jobs/details/57546097...,"Description\nJoin us, be part of more.\nWe're ...",2026-06-06 19:50:01,GB,42862-42862,"EXCEL, R",Hybrid,Unspecified,2026-06-01/2026-06-07,"Hamilton, South Lanarkshire",British Gas
1942,5761700959,ALTERNANCE - Data Management / Data Gouvernanc...,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['France', 'Ile-de-France', 'Hauts-de...",fr,NaN,NaN,NaN,https://www.adzuna.fr/details/5761700959?utm_m...,"Lieu : Meudon, France\nConstruisons ensemble u...",2026-06-12 22:48:43,FR,Not Found,"PYTHON, SQL, EXCEL, POWER BI, R",Onsite,Senior,2026-06-08/2026-06-14,"Meudon, Boulogne-Billancourt",Thales
2874,5756362901,Upskilling als Controller (m/w/d): Daten Manag...,{'__CLASS__': 'Adzuna::API::Response::Company'...,{'__CLASS__': 'Adzuna::API::Response::Location...,de,NaN,NaN,NaN,https://www.adzuna.de/details/5756362901?utm_m...,Smart Future Campus GmbH\nDer Smart Future Cam...,2026-06-08 17:26:59,DE,Not Found,"SQL, POWER BI, R, AZURE",Remote,Intern,2026-06-08/2026-06-14,"Niederbolheim, Kerpen",Smart Future Campus GmbH
4208,5745111292,Data Analyst with Chinese Japanese Korean Sour...,"{'display_name': 'Capgemini Polska', '__CLASS_...",{'__CLASS__': 'Adzuna::API::Response::Location...,pl,NaN,NaN,NaN,https://www.adzuna.pl/details/5745111292?utm_m...,"At Capgemini Invent, we believe difference dri...",2026-05-30 03:33:57,PL,€22,"EXCEL, R",Hybrid,Senior,2026-05-25/2026-05-31,"Warszawa, mazowieckie",Capgemini Polska
426,5610678551,Data Analyst - Stage - Paris,"{'display_name': 'Papernest', '__CLASS__': 'Ad...",{'__CLASS__': 'Adzuna::API::Response::Location...,fr,NaN,NaN,NaN,https://www.adzuna.fr/details/5610678551?utm_m...,"Depuis 2015, papernest révolutionne la gestion...",2026-02-02 20:02:26,FR,Not Found,"SQL, TABLEAU, R",Unspecified,Senior,2026-02-02/2026-02-08,"19ème Arrondissement, Paris",Papernest
211,5749924520,Data Engineer,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name

In [112]:
jobs_df.columns

Index(['id', 'title', 'company', 'location', 'country', 'salary_min',
       'salary_max', 'contract_type', 'redirect_url', 'description', 'created',
       'clean_country', 'clean_salary', 'extracted_skills', 'work_mode',
       'seniority_level', 'year_week', 'clean_city', 'clean_company'],
      dtype='object')

In [113]:
jobs_df_clean = jobs_df.copy()

In [114]:
jobs_df_clean = jobs_df_clean.drop(columns=
                       ["company", 
                        "location", 
                        "salary_min",
                        "salary_max",
                        "country",
                        "redirect_url"]
                       )
jobs_df_clean.head(5)

,id,title,contract_type,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,year_week,clean_city,clean_company
0,5753281217,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,2026-06-01/2026-06-07,Österreich,STRABAG BRVZ GMBH
1,5753281216,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,2026-06-01/2026-06-07,Spittal an der Drau,STRABAG BRVZ GMBH
2,5762094132,Data Analyst - MS PowerBI (m/w/d),NaN,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13 06:58:14,AT,Not Found,"SQL, R",Hybrid,Intern,2026-06-08/2026-06-14,"Wien, Österreich",Baustoff + Metall Gesellschaft m.b.H. Österrei...
3,5727781690,Data Analyst (m/w/d),permanent,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13 08:03:25,AT,Not Found,"SQL, R",Hybrid,Intern,2026-05-11/2026-05-17,"Josefstadt, Wien",ÖSW Österreichisches Siedlungswerk Gemeinnützi...
4,5758608143,Data Analyst,NaN,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10 17:18:19,AT,Not Found,"PYTHON, SQL, TABLEAU, R",Unspecified,Unspecified,2026-06-08/2026-06-14,Österreich,Hitapps


In [115]:
jobs_df_clean.nunique().sort_values(ascending=False)


id                  6718
created             5863
description         5592
title               4403
clean_company       3018
clean_salary        2027
clean_city          1378
extracted_skills     247
year_week            113
clean_country         10
seniority_level        5
work_mode              4
contract_type          2
dtype: int64

In [116]:
jobs_df_clean.describe(include="object")

,title,contract_type,description,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,year_week,clean_city,clean_company
count,6718,1700,6718,6718,6718,6718,6718,6718,6718,6718,6718
unique,4403,2,5592,10,2027,247,4,5,113,1378,3018
top,Data Analyst,permanent,Too Many Requests,FR,Not Found,R,Unspecified,Senior,2026-06-08/2026-06-14,"London, UK",Externatic
freq,325,998,465,2095,3420,1449,3395,3644,1279,550,235


In [117]:
jobs_df_clean.columns

Index(['id', 'title', 'contract_type', 'description', 'created',
       'clean_country', 'clean_salary', 'extracted_skills', 'work_mode',
       'seniority_level', 'year_week', 'clean_city', 'clean_company'],
      dtype='object')

In [118]:
jobs_df_clean["title"].unique()

array(['Data Analyst (m/w/d)', 'Data Analyst - MS PowerBI (m/w/d)',
       'Data Analyst', ..., 'Executive, Data Excellence',
       'Founding Product Designer', 'Business Analyst (Ref: 18005)'],
      shape=(4403,), dtype=object)

In [119]:
jobs_df_clean["contract_type"].unique()

array([nan, 'permanent', 'contract'], dtype=object)

In [120]:
jobs_df_clean["clean_country"].unique()

array(['AT', 'BE', 'FR', 'DE', 'IT', 'NL', 'PL', 'ES', 'CH', 'GB'],
      dtype=object)

In [121]:
jobs_df_clean["work_mode"].unique()

array(['Onsite', 'Hybrid', 'Unspecified', 'Remote'], dtype=object)

In [122]:
jobs_df_clean["seniority_level"].unique()

array(['Intern', 'Unspecified', 'Junior', 'Senior', 'Middle'],
      dtype=object)

In [123]:
jobs_df_clean["clean_company"].unique()

array(['STRABAG BRVZ GMBH',
       'Baustoff + Metall Gesellschaft m.b.H. Österreich Hauptsitz',
       'ÖSW Österreichisches Siedlungswerk Gemeinnützige Wohnungs AG',
       ..., 'ServiceTrade', 'WPP Media WPP', 'Fifth Dimension'],
      shape=(3018,), dtype=object)

In [124]:
jobs_df_clean["contract_type"].value_counts(dropna=False)

contract_type
NaN          5018
permanent     998
contract      702
Name: count, dtype: int64

In [125]:
jobs_df_clean["clean_country"].value_counts(dropna=False)

clean_country
FR    2095
GB    1968
DE     782
PL     591
NL     442
IT     271
ES     245
BE     191
CH      68
AT      65
Name: count, dtype: int64

In [126]:
jobs_df_clean["work_mode"].value_counts(dropna=False)

work_mode
Unspecified    3395
Hybrid         1717
Remote          954
Onsite          652
Name: count, dtype: int64

In [127]:
jobs_df_clean["seniority_level"].value_counts(dropna=False)

seniority_level
Senior         3644
Unspecified    1777
Intern         1106
Junior          186
Middle            5
Name: count, dtype: int64

In [128]:
jobs_df_clean["clean_company"].value_counts(dropna=False)

clean_company
Externatic                       235
Unknown                          154
Smart Future Campus GmbH         105
Collective.work                  105
ITOL Recruit                      88
                                ... 
Usercentrics                       1
FullStack Talents                  1
MCG Memmingen GmbH                 1
EFW – Elbe Flugzeugwerke GmbH      1
Fifth Dimension                    1
Name: count, Length: 3018, dtype: int64

In [129]:
jobs_df_clean.duplicated().sum()

np.int64(0)

In [130]:
jobs_df_clean["title_clean"] = (
    jobs_df_clean["title"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("-", " ", regex=False)
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [131]:
jobs_df_clean["title_clean"].value_counts().head(20)

title_clean
data analyst                                                                                                    479
senior data analyst                                                                                             137
data engineer                                                                                                   127
data analyst trainee                                                                                             61
upskilling als controller : daten management mit ki, azure & power bi – für den schnellen wiedereinstieg         39
analytics engineer                                                                                               37
referent der geschäftsführung : datenanalyse für unternehmenssteuerung & reporting mit sql, azure & power bi     33
deine chance: datenmanagement upskilling   sql, microsoft azure & power bi für den schnellen wiedereinstieg      33
data analyst f/h                                            

In [132]:
jobs_df_clean[["title", "title_clean"]].head(20)

,title,title_clean
0,Data Analyst (m/w/d),data analyst
1,Data Analyst (m/w/d),data analyst
2,Data Analyst - MS PowerBI (m/w/d),data analyst ms powerbi
3,Data Analyst (m/w/d),data analyst
4,Data Analyst,data analyst
5,(Junior) Data Analyst,data analyst
6,Senior Data Analyst:in,senior data analyst:in
7,Supply Chain Data Analyst*in,supply chain data analyst*in
8,Data Analyst (Gaming Industry),data analyst
9,Data Analyst / Data Manager (m/w/d),data analyst / data manager


In [133]:
jobs_df_clean["job_category"] = "other"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("data analyst", na=False),
    "job_category"
] = "data analyst"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("engineer", na=False),
    "job_category"
] = "data engineer"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("scientist", na=False),
    "job_category"
] = "data scientist"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("business", na=False),
    "job_category"
] = "business analyst"

In [134]:
jobs_df_clean["job_category"].value_counts().head(20)

job_category
other               2819
data analyst        2496
data engineer        754
business analyst     499
data scientist       150
Name: count, dtype: int64

In [135]:
jobs_df_clean["clean_salary"] = (
    jobs_df_clean["clean_salary"]
    .replace("not found", pd.NA)
    .astype("string")
    .str.replace(r"[€$£¥₹]", "", regex=True)
    .str.replace(r"\b(eur|usd|gbp)\b", "", regex=True, case=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [136]:
jobs_df_clean[["clean_salary"]].head(20)

,clean_salary
0,Not Found
1,Not Found
2,Not Found
3,Not Found
4,Not Found
5,Not Found
6,Not Found
7,Not Found
8,Not Found
9,Not Found


In [137]:
jobs_df_clean["clean_salary"] = (
    jobs_df_clean["clean_salary"]
    .astype("string")
    .str.replace(r"[€$£¥₹]", "", regex=True)
    .str.strip()
)

salary_split = jobs_df_clean["clean_salary"].str.split("-", n=1, expand=True)

jobs_df_clean["min_salary"] = salary_split[0].str.strip()
jobs_df_clean["max_salary"] = salary_split[1].str.strip()

jobs_df_clean.loc[
    jobs_df_clean["clean_salary"] == "not found",
    ["min_salary", "max_salary"]
] = "not found"

In [138]:
jobs_df_clean[["clean_salary", "min_salary", "max_salary"]].head(20)

,clean_salary,min_salary,max_salary
0,Not Found,Not Found,<NA>
1,Not Found,Not Found,<NA>
2,Not Found,Not Found,<NA>
3,Not Found,Not Found,<NA>
4,Not Found,Not Found,<NA>
5,Not Found,Not Found,<NA>
6,Not Found,Not Found,<NA>
7,Not Found,Not Found,<NA>
8,Not Found,Not Found,<NA>
9,Not Found,Not Found,<NA>


In [139]:

import numpy as np

jobs_df_clean["currency"] = np.where(
    jobs_df_clean["clean_country"].str.lower() == "pl",
    "PLN",
    np.where(
        jobs_df_clean["clean_country"].str.lower() == "gb",
        "GBP",
        "EUR"
    )
)

In [140]:
jobs_df_clean[["clean_country", "currency"]].sample(20)

,clean_country,currency
4619,ES,EUR
5955,GB,GBP
1945,FR,EUR
431,FR,EUR
387,FR,EUR
6717,GB,GBP
5560,GB,GBP
2558,DE,EUR
179,BE,EUR
85,BE,EUR


In [146]:
exchange_rate_map = {
    "EUR": 1.0,
    "PLN": 4.26,
    "GBP": 0.87
}

jobs_df_clean["exchange_rate"] = jobs_df_clean["currency"].map(exchange_rate_map)

In [148]:
min_num = pd.to_numeric(jobs_df_clean["min_salary"], errors="coerce")
max_num = pd.to_numeric(jobs_df_clean["max_salary"], errors="coerce")

jobs_df_clean["MIN_EUR"] = min_num * jobs_df_clean["exchange_rate"]
jobs_df_clean["MAX_EUR"] = max_num * jobs_df_clean["exchange_rate"]

In [149]:
jobs_df_clean

,id,title,contract_type,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,...,clean_city,clean_company,title_clean,job_category,min_salary,max_salary,currency,exchange_rate,MIN_EUR,MAX_EUR
0,5753281217,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,...,Österreich,STRABAG BRVZ GMBH,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
1,5753281216,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,R,Onsite,Intern,...,Spittal an der Drau,STRABAG BRVZ GMBH,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
2,5762094132,Data Analyst - MS PowerBI (m/w/d),NaN,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13 06:58:14,AT,Not Found,"SQL, R",Hybrid,Intern,...,"Wien, Österreich",Baustoff + Metall Gesellschaft m.b.H. Österrei...,data analyst ms powerbi,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
3,5727781690,Data Analyst (m/w/d),permanent,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13 08:03:25,AT,Not Found,"SQL, R",Hybrid,Intern,...,"Josefstadt, Wien",ÖSW Österreichisches Siedlungswerk Gemeinnützi...,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
4,5758608143,Data Analyst,NaN,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10 17:18:19,AT,Not Found,"PYTHON, SQL, TABLEAU, R",Unspecified,Unspecified,...,Österreich,Hitapps,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6713,5734024897,"Executive, Data Excellence",NaN,About WPP Media\nWPP is the trusted growth par...,2026-05-19 13:14:36,GB,29787-29787,"SQL, EXCEL, POWER BI, R",Hybrid,Senior,...,"London, UK",WPP Media WPP,"executive, data excellence",other,29787,29787,GBP,0.87,25914.69,25914.69
6714,5731913923,Principal Product Marketing Manager,contract,"Sprinklr is the definitive, AI-native platform...",2026-05-17 01:20:20,GB,76986-76986,"EXCEL, R, AWS",Unspecified,Senior,...,"London, UK",Sprinklr,principal product marketing manager,other,76986,76986,GBP,0.87,66977.82,66977.82
6715,5672309969,Founding Product Designer,NaN,The Opportunity\nFifth Dimension is the most p...,2026-03-19 20:25:48,GB,70612-70612,R,Hybrid,Senior,...,"Oxford Circus, Central London",Fifth Dimension,founding product designer,other,70612,70612,GBP,0.87,61432.44,61432.44
6716,5749733223,Business Analyst (Ref: 18005),permanent,About the job\nJob summary\nThis position is b...,2026-06-02 22:59:45,GB,51993-51993,"EXCEL, R",Remote,Senior,...,"London, UK",. Ministry of Justice,business analyst,business analyst,51993,51993,GBP,0.87,45233.91,45233.91


In [150]:
columns_to_keep = [
    "id",
    "title_clean",
    "job_category",
    "contract_type",
    "created",
    "description",
    "clean_country",
    "clean_salary",
    "extracted_skills",
    "work_mode",
    "seniority_level",
    "clean_company",
    "clean_city",
    "min_salary",
    "max_salary",
    "currency",
    "exchange_rate",
    "MIN_EUR",
    "MAX_EUR"
]

jobs_df_final = jobs_df_clean[columns_to_keep].copy()

jobs_df_final

,id,title_clean,job_category,contract_type,created,description,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,clean_company,clean_city,min_salary,max_salary,currency,exchange_rate,MIN_EUR,MAX_EUR
0,5753281217,data analyst,data analyst,NaN,2026-06-05 16:40:19,Bei STRABAG bauen rund 89.000 Menschen an mehr...,AT,Not Found,R,Onsite,Intern,STRABAG BRVZ GMBH,Österreich,Not Found,<NA>,EUR,1.00,<NA>,<NA>
1,5753281216,data analyst,data analyst,NaN,2026-06-05 16:40:19,Bei STRABAG bauen rund 89.000 Menschen an mehr...,AT,Not Found,R,Onsite,Intern,STRABAG BRVZ GMBH,Spittal an der Drau,Not Found,<NA>,EUR,1.00,<NA>,<NA>
2,5762094132,data analyst ms powerbi,data analyst,NaN,2026-06-13 06:58:14,Baustoff + Metall ist ein hoch spezialisierter...,AT,Not Found,"SQL, R",Hybrid,Intern,Baustoff + Metall Gesellschaft m.b.H. Österrei...,"Wien, Österreich",Not Found,<NA>,EUR,1.00,<NA>,<NA>
3,5727781690,data analyst,data analyst,permanent,2026-05-13 08:03:25,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,AT,Not Found,"SQL, R",Hybrid,Intern,ÖSW Österreichisches Siedlungswerk Gemeinnützi...,"Josefstadt, Wien",Not Found,<NA>,EUR,1.00,<NA>,<NA>
4,5758608143,data analyst,data analyst,NaN,2026-06-10 17:18:19,"What are you working on?\nGenres: Casual, Puzz...",AT,Not Found,"PYTHON, SQL, TABLEAU, R",Unspecified,Unspecified,Hitapps,Österreich,Not Found,<NA>,EUR,1.00,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6713,5734024897,"executive, data excellence",other,NaN,2026-05-19 13:14:36,About WPP Media\nWPP is the trusted growth par...,GB,29787-29787,"SQL, EXCEL, POWER BI, R",Hybrid,Senior,WPP Media WPP,"London, UK",29787,29787,GBP,0.87,25914.69,25914.69
6714,5731913923,principal product marketing manager,other,contract,2026-05-17 01:20:20,"Sprinklr is the definitive, AI-native platform...",GB,76986-76986,"EXCEL, R, AWS",Unspecified,Senior,Sprinklr,"London, UK",76986,76986,GBP,0.87,66977.82,66977.82
6715,5672309969,founding product designer,other,NaN,2026-03-19 20:25:48,The Opportunity\nFifth Dimension is the most p...,GB,70612-70612,R,Hybrid,Senior,Fifth Dimension,"Oxford Circus, Central London",70612,70612,GBP,0.87,61432.44,61432.44
6716,5749733223,business analyst,business analyst,permanent,2026-06-02 22:59:45,About the job\nJob summary\nThis position is b...,GB,51993-51993,"EXCEL, R",Remote,Senior,. Ministry of Justice,"London, UK",51993,51993,GBP,0.87,45233.91,45233.91


In [151]:
jobs_df_final.to_csv("jobs_clean.csv", index=False)